In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2

def start_video_stream():
  js = Javascript('''
    var video; var div = null; var stream; var captureCanvas; var labelElement;
    var pendingResolve = null; var shutdown = false;

    function removeDom() {
       stream.getVideoTracks()[0].stop();
       video.remove(); div.remove();
       video = null; div = null; stream = null; captureCanvas = null; labelElement = null;
    }

    function onAnimationFrame() {
      if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          // --- ส่วนที่ปรับปรุง: สลับข้างภาพให้เป็นกระจกเงาก่อนส่งไป Python ---
          var ctx = captureCanvas.getContext('2d');
          ctx.translate(640, 0);
          ctx.scale(-1, 1);
          ctx.drawImage(video, 0, 0, 640, 480);
          ctx.setTransform(1, 0, 0, 1, 0, 0); // Reset ค่ากลับ
          // --------------------------------------------------------
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve; pendingResolve = null; lp(result);
      }
    }

    async function createDom() {
      if (div !== null) return stream;
      div = document.createElement('div');
      div.style.border = '2px solid black'; div.style.padding = '3px';
      div.style.width = '100%'; div.style.maxWidth = '600px';
      document.body.appendChild(div);

      const modelOut = document.createElement('div');
      labelElement = document.createElement('span');
      labelElement.innerText = 'กำลังเปิดกล้อง...'; labelElement.style.fontWeight = 'bold';
      modelOut.appendChild(labelElement); div.appendChild(modelOut);

      video = document.createElement('video');
      video.style.display = 'block'; video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', ''); video.onclick = () => { shutdown = true; };

      // --- ส่วนที่ปรับปรุง: พลิกภาพวิดีโอบนหน้าจอเบราว์เซอร์ให้เป็นกระจกเงา ---
      video.style.transform = 'scaleX(-1)';
      // -------------------------------------------------------------

      stream = await navigator.mediaDevices.getUserMedia({video: { facingMode: "user"}}); // ใช้กล้องหน้า
      div.appendChild(video);

      const instruction = document.createElement('div');
      instruction.innerHTML = '<span style="color: red; font-weight: bold; cursor: pointer;">🛑 คลิกที่ภาพวิดีโอเพื่อหยุดกล้อง</span>';
      div.appendChild(instruction);

      video.srcObject = stream; await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640; captureCanvas.height = 480;
      window.requestAnimationFrame(onAnimationFrame);
      return stream;
    }

    async function stream_frame(label) {
      if (shutdown) { removeDom(); shutdown = false; return ''; }
      stream = await createDom();
      if (label !== "") labelElement.innerHTML = label;
      var result = await new Promise(function(resolve, reject) { pendingResolve = resolve; });
      shutdown = false; return {'img': result};
    }
    ''')
  display(js)

def get_video_frame(label):
  data = eval_js('stream_frame("{}")'.format(label))
  return data

def base64_to_cv2(js_reply):
  image_bytes = b64decode(js_reply.split(',')[1])
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  img = cv2.imdecode(jpg_as_np, flags=1)
  return img

In [ ]:
start_video_stream()
label_text = 'กล้องกำลังทำงาน... (ส่งภาพเข้า Python สำเร็จ)'

while True:
    # 1. รับภาพ 1 เฟรมจากวิดีโอ
    js_reply = get_video_frame(label_text)

    # 2. เช็คว่าผู้ใช้กดคลิกปิดกล้องไปแล้วหรือยัง
    if not js_reply:
        print("ปิดกล้องเรียบร้อย")
        break

    # 3. แปลงภาพที่ดึงมาให้กลายเป็น Array ของ OpenCV
    img_cv2 = base64_to_cv2(js_reply["img"])

    # =========================================================
    # ตอนนี้ภาพแต่ละเฟรมถูกเตรียมพร้อมไว้ในตัวแปร img_cv2 แล้ว
    # หากรันเซลล์นี้สำเร็จ วิดีโอจะขยับตามปกติ
    # (ใน Step ถัดไปของ Workshop เราจะเอา img_cv2 ส่งเข้าโมเดล AI ตรงจุดนี้)
    # =========================================================